# P1b — controlled multi-task model and latency (GPU)

* **base_sel_emo**: the base multi-task model retrained with the epoch selected on the **emotion NLL only**, the same criterion as the direct emotion CNN (3 seeds × 4 folds, ResNet-18, short).
* **Latency**: network latency of all configurations including MobileNetV3-S + Transformer, and the full pipeline on CPU (BlazeFace detector + crop + network; FaceLandmarker for the teacher side).

All runs already on Drive are skipped. *Runtime → Change runtime type → T4 GPU*, then run the cell; re-run after a disconnection.

In [ ]:
# ======================================================================
#  Pseudo-AU / RAVDESS — P1b (GPU): controlled multi-task model + latency
#  Pseudo-AU / RAVDESS — FULL COLAB PIPELINE IN A SINGLE CELL
#  Saved to Google Drive after EVERY step:
#    each actor downloaded + extracted -> features on Drive
#    baselines                         -> results on Drive
#    each training FOLD                -> saved on Drive (at most ~10-40 min of work lost)
#  Free GPU disconnected? Simply re-run this cell (later, or the next day):
#  everything already done is skipped.
# ======================================================================
# ============================ SETTINGS ============================
SEEDS       = [0, 1, 2]   # 3 seeds for the paper
EPOCHS      = 15
FOLDS       = 4
RUN_ABLATIONS = True      # tmp_none, tmp_tcn, loss_l1, loss_ccc, bb_mobilenet
RUN_VIT     = True        # ViT-B/16: very long, run last
RUN_BASE_SEL_EMO = True   # P1b: base multi-task model, epoch selected on emotion NLL only (as the direct model)
MEASURE_PIPELINE = True   # P1b: latency of MobileNetV3+Transformer and of the full pipeline (detector + crop + net)
RUN_VIT_CCC = False       # P2: ViT-B/16 + CCC+L1 (long, about one week of free GPU)
VIT_BATCH   = 8
# ==================================================================
# The cell adapts to the available hardware:
#  - WITHOUT GPU: downloads RAVDESS, extracts features, computes baselines, then stops.
#  - WITH GPU   : trains in the paper's priority order, then builds the report.
# After a disconnection, simply re-run the cell: everything resumes.

# ======================================================================
#  EXECUTION (do not modify below this line)
# ======================================================================
import os, sys, time, json, shutil, zipfile, subprocess, urllib.request
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline backend breaks matplotlib in subprocesses
from pathlib import Path

T0 = time.time()
def step(t): print("\n" + "=" * 72 + f"\n  {t}   [{(time.time()-T0)/60:.0f} min]\n" + "=" * 72, flush=True)

def run(cmd, cwd=None, env=None, must_succeed=True):
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if must_succeed and p.returncode != 0:
        raise RuntimeError(f"FAILED (code {p.returncode}): {cmd}\n-> The cause is in the lines above.")
    return p.returncode

def copy_safe(src, dst, tries=4):
    """Copy that tolerates transient Google Drive errors."""
    for k in range(tries):
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            return True
        except OSError:
            time.sleep(2 * (k + 1))
    return False

def mirror(src_dir, dst_dir, pattern='*'):
    """Copy missing (or different-size) files from src_dir to dst_dir."""
    n, fail = 0, 0
    for f in Path(src_dir).rglob(pattern):
        if not f.is_file():
            continue
        t = Path(dst_dir) / f.relative_to(src_dir)
        try:
            if t.exists() and t.stat().st_size == f.stat().st_size:
                continue
        except OSError:
            pass
        if copy_safe(f, t): n += 1
        else: fail += 1
    return n, fail

PY = sys.executable

# ------------------------------------------------------------------ 1. Drive + code
step("1. Google Drive and project code")
from google.colab import drive
drive.mount('/content/drive')
DRIVE   = Path('/content/drive/MyDrive')
PROJ_D  = DRIVE / 'ravdess_au_project'            # on Drive (persistent)
FEAT_D  = PROJ_D / 'data' / 'features'
RES_D   = PROJ_D / 'results'
PROJ    = Path('/content/ravdess_au_project')     # local copy (fast)
FEAT_L  = PROJ / 'data' / 'features'
RES_L   = PROJ / 'results'
RAW     = Path('/content/raw')

if not (PROJ_D / 'train_eval.py').exists():
    z = DRIVE / 'ravdess_au_project.zip'
    if not z.exists():
        from google.colab import files
        print("ravdess_au_project.zip not found in My Drive -> please upload it:")
        up = files.upload()
        z = Path('/content') / next(iter(up))
    with zipfile.ZipFile(z) as f:
        f.extractall(DRIVE)
assert (PROJ_D / 'train_eval.py').exists(), "train_eval.py not found in My Drive/ravdess_au_project"
PROJ.mkdir(exist_ok=True)
for f in PROJ_D.iterdir():                          # code + paper -> local
    if f.is_file():
        copy_safe(f, PROJ / f.name)
if (PROJ_D / 'paper').is_dir():
    mirror(PROJ_D / 'paper', PROJ / 'paper')
for p in (FEAT_D, RES_D, FEAT_L, RES_L, RAW):
    p.mkdir(parents=True, exist_ok=True)
os.chdir(PROJ)

# --- Fold-by-fold saving: patches the LOCAL copy of train_eval.py (Drive copy unchanged) ---
_te = (PROJ / 'train_eval.py').read_text()
if 'DRIVE_MIRROR' not in _te:
    A = """            au_true, au_pred, prob = [None] * V, [None] * V, np.zeros((V, 8))
            for fi, fold in enumerate(folds):
"""
    A2 = """            au_true, au_pred, prob = [None] * V, [None] * V, np.zeros((V, 8))
            import pickle, os, shutil as _sh
            _mir = os.environ.get("DRIVE_MIRROR")
            for fi, fold in enumerate(folds):
                _pf = Path(args.out) / name / f"seed{seed}_fold{fi}.pkl"
                if not _pf.exists() and _mir and (Path(_mir) / name / _pf.name).exists():
                    _pf.parent.mkdir(parents=True, exist_ok=True)
                    _sh.copy2(Path(_mir) / name / _pf.name, _pf)
                if _pf.exists():
                    with open(_pf, "rb") as _f:
                        _d = pickle.load(_f)
                    for j, v in enumerate(_d["te"]):
                        au_true[v], au_pred[v], prob[v] = _d["t"][j], _d["p"][j], _d["e"][j]
                    print(f"[resume] {name} seed{seed} fold{fi + 1} already done -> skipped", flush=True)
                    continue
"""
    B = """                print(f"    best epoch {best_ep}; test acc {np.mean(p_emo.argmax(1) == emo[te]):.3f}", flush=True)
"""
    B2 = B + """                _pf.parent.mkdir(parents=True, exist_ok=True)
                with open(_pf, "wb") as _f:
                    pickle.dump({"te": te, "t": [au_true[v] for v in te], "p": [au_pred[v] for v in te],
                                 "e": prob[te]}, _f)
                if _mir:
                    for _k in range(4):
                        try:
                            (Path(_mir) / name).mkdir(parents=True, exist_ok=True)
                            _sh.copy2(_pf, Path(_mir) / name / _pf.name)
                            print(f"    fold{fi + 1} saved to Drive", flush=True)
                            break
                        except OSError:
                            time.sleep(3)
"""
    assert A in _te and B in _te, "unexpected train_eval.py: fold-by-fold patch not applied"
    (PROJ / 'train_eval.py').write_text(_te.replace(A, A2).replace(B, B2))
    print("Fold-by-fold saving enabled")

# --- Clearer English figure labels: patches the LOCAL copy of make_report.py ---
_mr = (PROJ / 'make_report.py').read_text()
if 'Agreement with MediaPipe' not in _mr:
    _rep = [
        ('    ax.set_ylim(-0.1, 1.05)\n',
         '    ax.set_ylim(-0.1, 1.05)\n    ax.set_xlabel("Action Unit")\n    ax.set_ylabel("Agreement with MediaPipe pseudo-AU")\n'
         '    ax.set_title("Per-AU agreement (mean and 95% CI)", fontsize=9)\n'),
        ('ax.set_xticks(range(8), EMOTIONS, rotation=60, fontsize=7)',
         'ax.set_xticks(range(8), [e_.capitalize() for e_ in EMOTIONS], rotation=60, fontsize=7)'),
        ('ax.set_yticks(range(8), EMOTIONS, fontsize=7)',
         'ax.set_yticks(range(8), [e_.capitalize() for e_ in EMOTIONS], fontsize=7)'),
        ('ax.set_xlabel("predicted")', 'ax.set_xlabel("Predicted emotion")'),
        ('ax.set_ylabel("true")', 'ax.set_ylabel("True emotion")\n    ax.set_title("Emotion confusion matrix (row-normalised)", fontsize=9)'),
    ]
    for _a, _b in _rep:
        if _a in _mr:
            _mr = _mr.replace(_a, _b, 1)
        else:
            print("Note: figure label not patched:", _a.strip()[:50])
    (PROJ / 'make_report.py').write_text(_mr)
    print("English figure labels enabled")

# --- P1b: new experiments in the LOCAL copies of the scripts (Drive copies unchanged) ---
_te = (PROJ / 'train_eval.py').read_text()
if '"base_sel_emo"' not in _te:
    _a = '    "bb_vit": {"backbone": "vit_b_16"},\n'
    _b = '        crit = (cfg["lam_au"] > 0) * l1 + (cfg["lam_emo"] > 0) * nll\n'
    assert _a in _te and _b in _te, "unexpected train_eval.py: P1b patch not applied"
    _te = _te.replace(_a, _a + '    "base_sel_emo": {"select": "emo"},\n    "vit_ccc": {"backbone": "vit_b_16", "loss": "ccc"},\n')
    _te = _te.replace(_b, '        crit = nll if cfg.get("select") == "emo" else (cfg["lam_au"] > 0) * l1 + (cfg["lam_emo"] > 0) * nll\n')
    (PROJ / 'train_eval.py').write_text(_te)
_mr = (PROJ / 'make_report.py').read_text()
if '"base_sel_emo"' not in _mr:
    _a = '    "bb_vit": "backbone: ViT-B/16",\n'
    assert _a in _mr
    _mr = _mr.replace(_a, _a + '    "base_sel_emo": "base, epoch selected on emotion NLL",\n'
                                '    "vit_ccc": "ViT-B/16 + CCC+L1 (best combination)",\n')
    _a = '        if S is not None:\n            pt, bs = au_boot(S, a.boot)'
    if _a in _mr:
        _mr = _mr.replace(_a, "        if S is not None and n != 'direct_emotion':\n            pt, bs = au_boot(S, a.boot)")
    (PROJ / 'make_report.py').write_text(_mr)
_ml = (PROJ / 'measure_latency.py').read_text()
if '("mobilenet_v3_small", "transformer")' not in _ml:
    _a = '(\"mobilenet_v3_small\", \"tcn\"),'
    assert _a in _ml
    (PROJ / 'measure_latency.py').write_text(_ml.replace(_a, _a + ' ("mobilenet_v3_small", "transformer"),'))
print("P1b experiments enabled: base_sel_emo, vit_ccc; latency adds MobileNetV3-S + Transformer")

ON_GPU = subprocess.run([PY, '-c', 'import torch,sys; sys.exit(0 if torch.cuda.is_available() else 1)'],
                        capture_output=True).returncode == 0
gpu_name = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                          capture_output=True, text=True).stdout.strip() if ON_GPU else ""
print("Hardware:", f"GPU {gpu_name}" if ON_GPU else "CPU only (data preparation)",
      "|", os.cpu_count(), "vCPU")

# Resume: features and results already on Drive -> local disk
print("Restoring features from Drive...", flush=True)
TAR_D = PROJ_D / 'data' / 'features.tar'          # one big file = fast copy (vs 1440 small files)
n, fail = 0, 0
if TAR_D.exists() and len(list(FEAT_L.glob('*.npz'))) < 1440:
    t_ = time.time()
    if copy_safe(TAR_D, Path('/content/features.tar')):
        run(f'tar -xf /content/features.tar -C "{FEAT_L}"', must_succeed=False)
        Path('/content/features.tar').unlink(missing_ok=True)
        print(f"  archive restored in {(time.time() - t_) / 60:.1f} min")
n, fail = mirror(FEAT_D, FEAT_L, '*.npz'); mirror(FEAT_D, FEAT_L, 'index.csv')
print(f"  {len(list(FEAT_L.glob('*.npz')))} local feature files ({n} copied, {fail} Drive errors)")
mirror(RES_D, RES_L)

# Remove feature files truncated by a disconnection
(Path('/content') / 'check_npz.py').write_text(r'''
import sys, zipfile, numpy as np
from pathlib import Path
bad = 0
for f in sorted(Path(sys.argv[1]).glob('*.npz')):
    try:
        with np.load(f) as z: z['au'].shape
    except (zipfile.BadZipFile, EOFError, ValueError, KeyError, OSError):
        f.unlink(missing_ok=True); bad += 1
print(bad, "corrupted feature files removed")
''')
run(f'{PY} /content/check_npz.py "{FEAT_L}"')


MISSING = [a for a in range(1, 25) if len(list(FEAT_L.glob(f'01-*-{a:02d}.npz'))) < 58]
NEED_MP = bool(MISSING) or MEASURE_PIPELINE
print("Actors still to extract:", MISSING if MISSING else "none (all 1440 features present)")

# ------------------------------------------------------------------ 2. Isolated MediaPipe (only if needed)
if NEED_MP:
    step("2. MediaPipe in a separate environment (avoids numpy conflicts)")
    VENV, MP_PY = Path('/content/mpenv'), Path('/content/mpenv/bin/python')
    ok = MP_PY.exists() and subprocess.run([str(MP_PY), '-c', 'import mediapipe, cv2, pandas'],
                                           capture_output=True).returncode == 0
    if not ok:
        # a) venv with Colab's Python
        shutil.rmtree(VENV, ignore_errors=True)
        if run(f'{PY} -m venv "{VENV}"', must_succeed=False) != 0 or not MP_PY.exists():
            shutil.rmtree(VENV, ignore_errors=True)
            run(f'{PY} -m pip install -q virtualenv')
            run(f'{PY} -m virtualenv -q "{VENV}"')
        run(f'"{MP_PY}" -m pip install -q --upgrade pip', must_succeed=False)
        if run(f'"{MP_PY}" -m pip install -q "mediapipe>=0.10.14" pandas', must_succeed=False) != 0:
            # b) no MediaPipe wheel for this Python -> Python 3.12 via uv
            print("MediaPipe unavailable for this Python -> installing a separate Python 3.12 (uv)")
            shutil.rmtree(VENV, ignore_errors=True)
            run(f'{PY} -m pip install -q uv')
            run(f'{PY} -m uv venv --python 3.12 "{VENV}"')
            run(f'{PY} -m uv pip install --python "{MP_PY}" "mediapipe>=0.10.14" pandas')
    run(f'"{MP_PY}" -c "import mediapipe, numpy; print(\'mediapipe\', mediapipe.__version__, \'| numpy\', numpy.__version__)"')

else:
    step("2. MediaPipe not needed: all features are already extracted -> skipped")

# ------------------------------------------------------------------ 3. Data, actor by actor
step("3. Download + MediaPipe extraction, actor by actor (saved to Drive)")
MODEL = Path('/content/face_landmarker.task')
new_feat = False
for a in range(1, 25):
    have = len(list(FEAT_L.glob(f'01-*-{a:02d}.npz')))
    if have >= 58:
        continue
    adir = RAW / f'Actor_{a:02d}'
    if not (adir.exists() and len(list(adir.glob('01-*.mp4'))) >= 60):
        url = f'https://zenodo.org/records/1188976/files/Video_Speech_Actor_{a:02d}.zip?download=1'
        zp = Path('/content') / f'actor_{a:02d}.zip'
        for k in range(4):
            try:
                print(f"Actor {a:02d}: downloading...", flush=True)
                urllib.request.urlretrieve(url, zp)
                with zipfile.ZipFile(zp) as z:
                    for m in z.namelist():
                        if Path(m).name.startswith('01-') and m.endswith('.mp4'):   # modality 01 (audio-visual) only
                            adir.mkdir(parents=True, exist_ok=True)
                            (adir / Path(m).name).write_bytes(z.read(m))
                break
            except Exception as e:
                print(f"  error {e} -> retrying in 30 s", flush=True); time.sleep(30)
        zp.unlink(missing_ok=True)
    if not adir.exists():
        raise RuntimeError(f"Actor {a:02d} not downloaded (Zenodo unavailable?). Try again later.")
    print(f"Actor {a:02d}: MediaPipe extraction of {len(list(adir.glob('01-*.mp4')))} videos...", flush=True)
    run(f'"{MP_PY}" extract_features.py --raw "{adir}" --out "{FEAT_L}" --workers {max(1, os.cpu_count())} '
        f'--model "{MODEL}"', cwd=PROJ)
    n, fail = mirror(FEAT_L, FEAT_D, f'01-*-{a:02d}.npz')
    print(f"  -> {n} feature files saved to Drive" + (f" ({fail} errors, retried on next run)" if fail else ""))
    shutil.rmtree(adir, ignore_errors=True)
    new_feat = True

if new_feat or not (FEAT_L / 'index.csv').exists():
    Path('/content/build_index.py').write_text(r'''
import sys, numpy as np, pandas as pd
from pathlib import Path
d = Path(sys.argv[1]); rows = []
for f in sorted(d.glob("*.npz")):
    z = np.load(f); n = int(z["au"].shape[0])
    rows.append(dict(file=f.name, actor=int(z["actor"]), emotion=int(z["emotion"]), vocal=int(z["vocal"]),
                     intensity=int(z["intensity"]), n_det=n, n_total=int(z["total"]),
                     det_rate=n / max(int(z["total"]), 1)))
pd.DataFrame(rows).to_csv(d / "index.csv", index=False)
print(len(rows), "videos indexed")
''')
    run(f'{PY} /content/build_index.py "{FEAT_L}"')
    copy_safe(FEAT_L / 'index.csv', FEAT_D / 'index.csv')
# Pack all features into one archive on Drive so the next session restores them in ~1 min
if len(list(FEAT_L.glob('*.npz'))) >= 1400 and (new_feat or not TAR_D.exists()):
    print("Packing features into a single archive on Drive (one time only)...", flush=True)
    run(f'tar -cf /content/features.tar -C "{FEAT_L}" .')
    print("  archive saved" if copy_safe(Path('/content/features.tar'), TAR_D) else "  archive not saved (Drive error), will retry next run")
    Path('/content/features.tar').unlink(missing_ok=True)

run(f'''{PY} -c "
import pandas as pd
idx = pd.read_csv('{FEAT_L}/index.csv')
print(len(idx), 'videos |', idx.actor.nunique(), 'actors | mean detection rate', round(idx.det_rate.mean(), 3),
      '| dropped (< 8 frames):', int((idx.n_det < 8).sum()))
"''')

# ------------------------------------------------------------------ 4. Baselines
step("4. Baselines")
if not (RES_L / 'lr_blend' / 'seed0.npz').exists():
    run(f'{PY} baselines.py --feat "{FEAT_L}" --out "{RES_L}"', cwd=PROJ)
    mirror(RES_L, RES_D)
else:
    print("Already computed -> skipped")

def report():
    if not list((RES_L / 'base').glob('seed*.npz')):
        print("'base' not finished yet: no report."); return
    run(f'{PY} make_report.py --results "{RES_L}" --out "{PROJ}/paper/generated"', cwd=PROJ, must_succeed=False)
    mirror(PROJ / 'paper' / 'generated', PROJ_D / 'paper' / 'generated')
    tex = PROJ / 'paper' / 'generated' / 'numbers.tex'
    if tex.exists(): print(tex.read_text())
    if run('which pdftoppm >/dev/null 2>&1 || apt-get -qq install -y poppler-utils >/dev/null 2>&1',
           must_succeed=False) == 0:
        from IPython.display import display, Image
        for name in ['fig_per_au.pdf', 'fig_emo_cm.pdf']:
            p = PROJ / 'paper' / 'generated' / name
            if p.exists():
                run(f'pdftoppm -png -r 110 -singlefile "{p}" "/content/{p.stem}"', must_succeed=False)
                if Path(f'/content/{p.stem}.png').exists():
                    display(Image(f'/content/{p.stem}.png'))

if not ON_GPU:
    print("\n" + "*" * 72)
    print("  DATA PREPARATION DONE (features + baselines on Drive).")
    print("  Next: Runtime -> Change runtime type -> T4 GPU,")
    print("  then re-run this cell. Training will start directly.")
    print("*" * 72)
else:
    # -------------------------------------------------------------- 5. Training by priority
    step(f"5. Training on {gpu_name}, in priority order")
    jobs = []
    for s in SEEDS:                                   # 1) core results of the paper
        jobs += [('base', s), ('direct_emotion', s)]
    if RUN_ABLATIONS:                                 # 2) ablations, seed by seed
        for s in SEEDS:
            jobs += [(e, s) for e in ['tmp_tcn', 'tmp_none', 'loss_l1', 'loss_ccc', 'bb_mobilenet']]
    if RUN_VIT:                                       # 3) ViT last
        jobs += [('bb_vit', s) for s in SEEDS]
    if RUN_BASE_SEL_EMO:                              # P1b) controlled multi-task vs direct comparison
        jobs = [('base_sel_emo', s) for s in SEEDS] + jobs
    if RUN_VIT_CCC:                                   # P2) best-factor combination
        jobs += [('vit_ccc', s) for s in SEEDS]

    latency_done = (RES_L / 'latency.json').exists()
    for i, (e, s) in enumerate(jobs, 1):
        if (RES_L / e / f'seed{s}.npz').exists():
            continue
        batch = VIT_BATCH if e in ('bb_vit', 'vit_ccc') else 16
        print(f"\n>>> [{i}/{len(jobs)}] {e} seed{s} (batch {batch})", flush=True)
        t = time.time()
        run_env = dict(os.environ, DRIVE_MIRROR=str(RES_D), PYTHONUNBUFFERED='1')
        run(f'{PY} train_eval.py --feat "{FEAT_L}" --out "{RES_L}" --exps {e} --seeds {s} '
            f'--folds {FOLDS} --epochs {EPOCHS} --batch {batch}', cwd=PROJ, env=run_env)
        n, fail = mirror(RES_L, RES_D)
        print(f"<<< {e} seed{s} finished in {(time.time() - t) / 60:.0f} min, saved to Drive", flush=True)
        if not latency_done and e == 'direct_emotion' and s == SEEDS[-1]:
            step("Latency (measured as soon as the core results are done)")
            run(f'{PY} measure_latency.py --out "{RES_L}/latency.json"', cwd=PROJ, must_succeed=False)
            (RES_L / 'hardware.txt').write_text(
                f"GPU: {gpu_name} (Google Colab)\n"
                + subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout
                + f"{os.cpu_count()} vCPU\n")
            mirror(RES_L, RES_D); latency_done = True
            report()

    if not latency_done:
        run(f'{PY} measure_latency.py --out "{RES_L}/latency.json"', cwd=PROJ, must_succeed=False)
        mirror(RES_L, RES_D)

    if MEASURE_PIPELINE:
        step("P1b. Network latency (all configurations, incl. MobileNetV3-S + Transformer)")
        run(f'{PY} measure_latency.py --out "{RES_L}/latency_v2.json"', cwd=PROJ, must_succeed=False)
        step("P1b. Full-pipeline latency on CPU: face detection / landmarks + crop (one RAVDESS actor)")
        vids = sorted(RAW.rglob('01-*.mp4'))[:10]
        if not vids:
            zp = Path('/content/actor_01.zip')
            urllib.request.urlretrieve('https://zenodo.org/records/1188976/files/Video_Speech_Actor_01.zip?download=1', zp)
            with zipfile.ZipFile(zp) as z:
                for m in z.namelist():
                    if Path(m).name.startswith('01-') and m.endswith('.mp4'):
                        (RAW / 'lat').mkdir(parents=True, exist_ok=True); (RAW / 'lat' / Path(m).name).write_bytes(z.read(m))
            zp.unlink(missing_ok=True)
            vids = sorted(RAW.rglob('01-*.mp4'))[:10]
        Path('/content/pipe_lat.py').write_text(r"""
import sys, time, json, urllib.request
import cv2, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mpp
from mediapipe.tasks.python import vision
cv2.setNumThreads(1)
LM = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
FD = 'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite'
for u, f in ((LM, '/content/lm.task'), (FD, '/content/fd.tflite')):
    try: open(f, 'rb').close()
    except OSError: urllib.request.urlretrieve(u, f)
lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mpp.BaseOptions(model_asset_path='/content/lm.task'), running_mode=vision.RunningMode.IMAGE,
    output_face_blendshapes=True, num_faces=1))
fd = vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
    base_options=mpp.BaseOptions(model_asset_path='/content/fd.tflite'), running_mode=vision.RunningMode.IMAGE))
t_lm, t_fd, t_crop, n, found = [], [], [], 0, 0
for path in sys.argv[1:]:
    cap = cv2.VideoCapture(path); i = 0
    while True:
        ok, bgr = cap.read()
        if not ok: break
        if i % 2 == 0:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            t = time.perf_counter(); r1 = lm.detect(img); t_lm.append(time.perf_counter() - t)
            t = time.perf_counter(); r2 = fd.detect(img); t_fd.append(time.perf_counter() - t)
            if r2.detections:
                b = r2.detections[0].bounding_box; found += 1
                t = time.perf_counter()
                cx, cy, s = b.origin_x + b.width / 2, b.origin_y + b.height / 2, max(b.width, b.height) * 1.25
                h, w = rgb.shape[:2]
                crop = rgb[int(max(cy - s / 2, 0)):int(min(cy + s / 2, h)), int(max(cx - s / 2, 0)):int(min(cx + s / 2, w))]
                crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_AREA)
                t_crop.append(time.perf_counter() - t)
            n += 1
        i += 1
    cap.release()
ms = lambda v: 1000 * float(np.median(v[5:])) if len(v) > 5 else float('nan')
res = dict(frames=n, face_found=found, landmarker_ms=ms(t_lm), face_detector_ms=ms(t_fd), crop_resize_ms=ms(t_crop),
           input_resolution=f"{int(rgb.shape[1])}x{int(rgb.shape[0])}", note='median per processed frame, CPU, 1 thread for OpenCV')
json.dump(res, open(sys.argv[0].replace('.py', '.json'), 'w'), indent=1); print(res)
""")
        run(f'"{MP_PY}" /content/pipe_lat.py ' + ' '.join(f'"{v}"' for v in vids), must_succeed=False)
        if Path('/content/pipe_lat.json').exists():
            shutil.copy2('/content/pipe_lat.json', RES_L / 'pipeline_latency.json')
            pl = json.load(open('/content/pipe_lat.json'))
            lat = json.load(open(RES_L / 'latency_v2.json')) if (RES_L / 'latency_v2.json').exists() else {}
            rows = []
            for k, v in lat.items():
                dev, cfg = k.split('/')
                rows.append((dev, cfg, v['ms_per_frame_amortised'],
                             pl['face_detector_ms'] + pl['crop_resize_ms'] + v['ms_per_frame_amortised'] if dev == 'cpu' else float('nan')))
            print("\nFull student pipeline on CPU = BlazeFace detector + crop/resize + network (ms/frame):")
            for r in rows: print(f"  {r[0]:5s} {r[1]:32s} network {r[2]:8.2f}   full CPU pipeline {r[3]:8.2f}")
            print(f"  Teacher side: FaceLandmarker (detection + mesh + blendshapes) = {pl['landmarker_ms']:.2f} ms/frame on CPU")
        mirror(RES_L, RES_D)

    step("6. Progress")
    for e in ['const', 'lr_gt_au', 'lr_blend', 'base', 'direct_emotion', 'tmp_tcn', 'tmp_none',
              'loss_l1', 'loss_ccc', 'bb_mobilenet', 'bb_vit', 'base_sel_emo', 'vit_ccc']:
        done = sorted(p.stem for p in (RES_L / e).glob('seed*.npz')) if (RES_L / e).exists() else []
        print(f"  {e:15s} {', '.join(done) if done else '--'}")

    step("7. Tables, figures and numbers for the paper")
    report()
    run(f'cd "{PROJ}" && zip -qr /content/ravdess_results_for_paper.zip paper results -x "*.npz"')
    copy_safe(Path('/content/ravdess_results_for_paper.zip'), PROJ_D / 'ravdess_results_for_paper.zip')
    print("\nDONE. The results zip is in My Drive/ravdess_au_project/.")

print(f"\nTotal time: {(time.time() - T0) / 60:.0f} min")
